In [15]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json


In [16]:
if platform.system() == 'Windows':
    os.environ['PYSPARK_PYTHON'] = sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [17]:
builder = SparkSession \
    .builder \
    .appName("Data with Nikk the Greek Spark Session") \
    .master("local[4]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [18]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [19]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [20]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze" 
}

In [21]:
@F.udf(returnType="STRING")
def get_properties(url):
   json_request = requests.get(url).json()
   return json.dumps(json_request["result"]["properties"])

In [22]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("properties", get_properties(F.col("url")))
    
bronze_instance = StarWarsBronze(spark, **options)


In [23]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [24]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-01-03 04:05:...|           Boba Fett| 22|https://www.swapi...|{"height": "183",...|
|2025-01-03 04:05:...|               IG-88| 23|https://www.swapi...|{"height": "200",...|
|2025-01-03 04:05:...|               Bossk| 24|https://www.swapi...|{"height": "190",...|
|2025-01-03 04:05:...|    Lando Calrissian| 25|https://www.swapi...|{"height": "177",...|
|2025-01-03 04:05:...|               Lobot| 26|https://www.swapi...|{"height": "175",...|
|2025-01-03 04:05:...|              Ackbar| 27|https://www.swapi...|{"height": "180",...|
|2025-01-03 04:05:...|          Mon Mothma| 28|https://www.swapi...|{"height": "150",...|
|2025-01-03 04:05:...|        Arvel Crynyd| 29|https://www.swapi...|{"height": "unkno..

In [25]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show(truncate=False)

No. Rows: 60
+--------------------------+-----------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name       |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+-----------+---+-----------

In [26]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [27]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [28]:
#with load filter and transformation
class StarWarsSilver(silver.Silver):
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("uid <= '25'")
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
    
silver_instance = StarWarsSilver(spark, **options)

In [29]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute("people", "planets")

In [30]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 17
+--------------------------+-------------------------+---------------------+---+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS              |name                 |uid|url                                 |properties                                                                                                                                                                                                                |
+--------------------------+-------------------------+---------------------+---+------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [31]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show(100, truncate=False)

No. Rows: 18
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_SilverTS               |LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                       |
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2025-01-03 04:05:

In [32]:
#without load filter 
silver_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [33]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-01-03 04:05:...|2025-01-03 04:05:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|               C-3PO|  2|https://www.swapi...|{167, 75, n/a, go...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|               R2-D2|  3|https://www.swapi...|{96, 32, n/a, whi...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|         Darth Vader|  4|https://www.swapi...|{202, 136, none, ...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|         Leia Organa|  5|https://www.swapi...|{150, 49, brown, ...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|           Owen Lars|  6|https://www.swapi...|{178, 120,

In [34]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 60
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|2025-01-03 04:05:...|2025-01-03 04:05:...|       Mygeeto| 16|https://www.swapi...|{10088, 12, 167, ...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|       Felucia| 17|https://www.swapi...|{9100, 34, 231, 0...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|Cato Neimoidia| 18|https://www.swapi...|{0, 25, 278, 1 st...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|     Saleucami| 19|https://www.swapi...|{14920, 26, 392, ...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|       Stewjon| 20|https://www.swapi...|{0, unknown, unkn...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|        Eriadu| 21|https://www.swapi...|{13490, 24, 360, ...|
|2025-01-03 04:05:...|2025-01-03 04:05:...

# 3 Replace Where

In [35]:
class StarWarsSilver(silver.Silver): 
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("uid > '0'")
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
    
    def get_replace_condition(self, sdf: DataFrame, table: str) -> str:
        return "uid > 0"
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.load(filter="custom").transform().write(mode="replace").execute("people", "planets")

In [36]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|2025-01-03 04:05:...|2025-01-03 04:05:...|        Cliegg Lars| 62|https://www.swapi...|{183, unknown, br...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|  Poggle the Lesser| 63|https://www.swapi...|{183, 80, none, g...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|    Luminara Unduli| 64|https://www.swapi...|{170, 56.2, black...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|      Barriss Offee| 65|https://www.swapi...|{166, 50, black, ...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|              Dormé| 66|https://www.swapi...|{165, unknown, br...|
|2025-01-03 04:05:...|2025-01-03 04:05:...|              Dooku| 67|https://www.swapi...|{193, 80, white, ..

# 4 Append

In [37]:
class StarWarsSilver(silver.Silver): 
    def custom_filter(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("custom == 'custom'")
    
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write().execute("people", "planets") #default mode is append

In [38]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 164
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+--------------------+-------------------+---+--------------------+--------------------+
|2025-01-03 04:06:...|2025-01-03 04:05:...|        Cliegg Lars| 62|https://www.swapi...|{183, unknown, br...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|  Poggle the Lesser| 63|https://www.swapi...|{183, 80, none, g...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|    Luminara Unduli| 64|https://www.swapi...|{170, 56.2, black...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Barriss Offee| 65|https://www.swapi...|{166, 50, black, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|              Dormé| 66|https://www.swapi...|{165, unknown, br...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|              Dooku| 67|https://www.swapi...|{193, 80, white, .

# 5 Merge

In [39]:
class StarWarsSilver(silver.Silver): 
    def custom_transform(self, sdf: DataFrame, table: str) -> DataFrame:
        sdf = sdf.withColumn("properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table]))
        sdf = sdf.withColumn("uid", F.col("uid").cast("int"))
        return sdf
      
    def get_delta_merge_builder(self, sdf: DataFrame, delta_table: DeltaTable) -> DeltaMergeBuilder:
        merge_condition = "target.url = source.url" 
        builder = delta_table.alias("target").merge(sdf.alias("source"), merge_condition)
        builder = builder.whenMatchedUpdateAll()
        builder = builder.whenNotMatchedInsertAll()
        return builder
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.load().transform().write(mode="merge").execute("people", "planets")

In [40]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 164
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+--------------------+---+--------------------+--------------------+
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Luke Skywalker|  1|https://www.swapi...|{172, 77, blond, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Obi-Wan Kenobi| 10|https://www.swapi...|{182, 77, auburn,...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84, blond, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|    Anakin Skywalker| 11|https://www.swapi...|{188, 84,

In [41]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 120
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|         LH_SilverTS|         LH_BronzeTS|          name|uid|                 url|          properties|
+--------------------+--------------------+--------------+---+--------------------+--------------------+
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Tatooine|  1|https://www.swapi...|{10465, 23, 304, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|        Kamino| 10|https://www.swapi...|{19720, 27, 463, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:...|      Geonosis| 11|https://www.swapi...|{11370, 30, 256, ...|
|2025-01-03 04:06:...|2025-01-03 04:05:..

# 6 Clean Up

In [42]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]